# Finding hyperparameters for :-


1.   No. of hidden layers
2.   Size of the hidden layer



In [56]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [57]:
import pandas as pd
import numpy as np
df = pd.read_csv('fashion-mnist_train.csv')
df.head(3)

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,2,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,9,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,6,0,0,0,0,0,0,0,5,0,...,0.0,0.0,0.0,30.0,43.0,0.0,0.0,0.0,0.0,0.0


In [58]:
from sklearn.model_selection import train_test_split
X = df.drop('label', axis=1)
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [59]:
X_train = X_train/255
X_test = X_test/255

In [60]:
X_train = X_train.to_numpy()
X_test = X_test.to_numpy()
y_train = y_train.to_numpy()
y_test = y_test.to_numpy()

In [61]:
import torch
X_train = torch.from_numpy(X_train).to(torch.float32)
X_test = torch.from_numpy(X_test).to(torch.float32)
y_train = torch.from_numpy(y_train).to(torch.long)
y_test = torch.from_numpy(y_test).to(torch.long)

In [62]:
from torch.utils.data import Dataset, DataLoader
class CustomDataset(Dataset):
  def __init__(self, features, labels):
    self.features = features
    self.labels = labels

  def __len__(self):
    return len(self.features)

  def __getitem__(self, index):
    return self.features[index], self.labels[index]

In [63]:
train_ds = CustomDataset(X_train, y_train)
test_ds = CustomDataset(X_test, y_test)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=32, shuffle=False, pin_memory=True)

In [64]:
import torch
import torch.nn as nn

class Neural_Network(nn.Module):
  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer):
    super().__init__()
    layers = []
    for i in range(num_hidden_layers):
      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(0.3))
      input_dim = neurons_per_layer
    layers.append(nn.Linear(neurons_per_layer, output_dim))
    self.model = nn.Sequential(*layers)

  def forward(self, num_features):
    return self.model(num_features)

In [65]:
# Objective function
def objective(trial):
  # Hyperparameter values
  num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 5)
  neurons_per_layer = trial.suggest_int('neurons_per_layer', 8, 128, step=8)

  # model init
  input_dim = 784
  output_dim = 10

  model = Neural_Network(input_dim, output_dim, num_hidden_layers, neurons_per_layer)
  model.to(device)

  # params init
  learning_rate = 0.1
  epochs = 10

  # optimizer selection
  loss_function = nn.CrossEntropyLoss()
  optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=1e-4)

  # training loop
  for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)
      y_pred = model(batch_features)
      loss = loss_function(y_pred, batch_labels)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

  # evaluation
  model.eval()
  total = 0
  correct = 0

  with torch.no_grad():
    for batch_features, batch_labels in test_loader:
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)
      y_pred = model(batch_features)
      _, predicted = torch.max(y_pred, 1)
      total += batch_labels.shape[0]
      correct += (predicted == batch_labels).sum().item()
      accuracy = correct/total
  return accuracy

In [66]:
!pip install optuna

In [67]:
import optuna
study = optuna.create_study(direction='maximize')

[I 2026-01-08 03:49:21,603] A new study created in memory with name: no-name-d2f60010-fd7d-4a43-80de-794922355e4c


In [68]:
study.optimize(objective, n_trials=10)

[I 2026-01-08 03:50:00,265] Trial 0 finished with value: 0.5721932289575806 and parameters: {'num_hidden_layers': 2, 'neurons_per_layer': 120}. Best is trial 0 with value: 0.5721932289575806.
[I 2026-01-08 03:50:29,714] Trial 1 finished with value: 0.800080289040546 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 120}. Best is trial 1 with value: 0.800080289040546.
[I 2026-01-08 03:50:58,926] Trial 2 finished with value: 0.8317944600562023 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 80}. Best is trial 2 with value: 0.8317944600562023.
[I 2026-01-08 03:51:34,786] Trial 3 finished with value: 0.7808109193095143 and parameters: {'num_hidden_layers': 4, 'neurons_per_layer': 32}. Best is trial 2 with value: 0.8317944600562023.
[I 2026-01-08 03:51:55,941] Trial 4 finished with value: 0.5770105713903385 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 56}. Best is trial 2 with value: 0.8317944600562023.
[I 2026-01-08 03:52:17,035] Trial 5 finished 

In [69]:
study.best_value

0.8722066104643383

In [70]:
study.best_params

{'num_hidden_layers': 1, 'neurons_per_layer': 88}

# Finding hyperparameters for :-


1.  No. of hidden layers
2.  Size of the hidden layer
3.  epochs
4. learning rate
5. dropout rate
6. batch size
7. optimizer
8. weight decay



In [71]:
train_ds = CustomDataset(X_train, y_train)
test_ds = CustomDataset(X_test, y_test)

In [72]:
import torch
import torch.nn as nn

class Neural_Network(nn.Module):
  def __init__(self, input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate):
    super().__init__()
    layers = []
    for i in range(num_hidden_layers):
      layers.append(nn.Linear(input_dim, neurons_per_layer))
      layers.append(nn.BatchNorm1d(neurons_per_layer))
      layers.append(nn.ReLU())
      layers.append(nn.Dropout(dropout_rate))
      input_dim = neurons_per_layer
    layers.append(nn.Linear(neurons_per_layer, output_dim))
    self.model = nn.Sequential(*layers)

  def forward(self, num_features):
    return self.model(num_features)

In [73]:
# Objective function
def objective(trial):
  # Hyperparameter values
  num_hidden_layers = trial.suggest_int('num_hidden_layers', 1, 5)
  neurons_per_layer = trial.suggest_int('neurons_per_layer', 8, 128, step=8)
  epochs = trial.suggest_int('epochs', 10, 30, step=10)
  learning_rate = trial.suggest_float('learning_rate', 1e-5, 1e-1, log=True)
  dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5, step=0.1)
  batch_size = trial.suggest_categorical('batch_size', [16, 32, 64, 128])
  optimizer_name = trial.suggest_categorical('optimizer', ['Adam', 'SGD', 'RMSprop'])
  weight_decay = trial.suggest_float('weight_decay', 1e-5, 1e-3, log=True)

  train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
  test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

  # model init
  input_dim = 784
  output_dim = 10

  model = Neural_Network(input_dim, output_dim, num_hidden_layers, neurons_per_layer, dropout_rate)
  model.to(device)

  # optimizer selection
  loss_function = nn.CrossEntropyLoss()

  if optimizer_name == 'Adam':
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  elif optimizer_name == 'SGD':
    optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
  else:
    optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

  # training loop
  for epoch in range(epochs):
    for batch_features, batch_labels in train_loader:
      batch_features = batch_features.to(device)
      batch_labels = batch_labels.to(device)
      y_pred = model(batch_features)
      loss = loss_function(y_pred, batch_labels)
      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    # evaluation
    model.eval()
    total = 0
    correct = 0

    with torch.no_grad():
      for batch_features, batch_labels in test_loader:
        batch_features = batch_features.to(device)
        batch_labels = batch_labels.to(device)
        y_pred = model(batch_features)
        _, predicted = torch.max(y_pred, 1)
        total += batch_labels.shape[0]
        correct += (predicted == batch_labels).sum().item()

    accuracy = correct/total
    trial.report(accuracy, epoch)
    if trial.should_prune():
      raise optuna.exceptions.TrialPruned()

  return accuracy

In [74]:
import optuna
study = optuna.create_study(direction='maximize')

[I 2026-01-08 03:54:53,959] A new study created in memory with name: no-name-940e7a19-ba4f-4de1-9be9-d8c23ff53293


In [76]:
study.optimize(objective, n_trials=10)

[I 2026-01-08 03:57:00,299] Trial 1 finished with value: 0.10250234176368259 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 80, 'epochs': 20, 'learning_rate': 0.05753258487560896, 'dropout_rate': 0.30000000000000004, 'batch_size': 16, 'optimizer': 'RMSprop', 'weight_decay': 9.590568454461671e-05}. Best is trial 1 with value: 0.10250234176368259.
[I 2026-01-08 03:57:20,673] Trial 2 finished with value: 0.6771042419376422 and parameters: {'num_hidden_layers': 1, 'neurons_per_layer': 16, 'epochs': 20, 'learning_rate': 8.72574324632924e-05, 'dropout_rate': 0.1, 'batch_size': 128, 'optimizer': 'SGD', 'weight_decay': 0.0003144222661973292}. Best is trial 2 with value: 0.6771042419376422.
[I 2026-01-08 04:00:14,009] Trial 3 finished with value: 0.8077077478924127 and parameters: {'num_hidden_layers': 3, 'neurons_per_layer': 16, 'epochs': 30, 'learning_rate': 0.027613116626829706, 'dropout_rate': 0.2, 'batch_size': 16, 'optimizer': 'Adam', 'weight_decay': 2.5720168618439538e-05}

In [77]:
study.best_value

0.8637762612070119

In [78]:
study.best_params

{'num_hidden_layers': 5,
 'neurons_per_layer': 40,
 'epochs': 30,
 'learning_rate': 0.004296664603251061,
 'dropout_rate': 0.5,
 'batch_size': 128,
 'optimizer': 'Adam',
 'weight_decay': 0.00011890746189429898}